## code to translate German to English and vice versa with deepL

In [9]:
import pandas as pd
from pprint import pprint
import os
import ast
import deepl
from tqdm import tqdm
import json
import requests
import matplotlib.pyplot as plt

In [10]:
# get connection to deepl api
deepl_key = open("/Users/sofiaromero/Documents/DeepL.txt").read()
deepl_client = deepl.DeepLClient(deepl_key)

In [11]:
# api functions 

def generate_glossaries(dict_path_l, s_lang_l, t_lang_l, auth_key):
    url = "https://api.deepl.com/v3/glossaries"
    header= {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
        "Content-Type": "application/json"
    }    
    glossaries = []
    for i,d in enumerate(dict_path_l):
        payload = {
            "name": f"glossary{i}",
            "dictionaries": [{
                # TODO: get languages from file name
                "source_lang": s_lang_l[i],
                "target_lang": t_lang_l[i],
                "entries": open(d,"r").read(),
                "entries_format": "csv"}
            ]}
        resp = requests.post(url, headers=header, json=payload)
        print(resp.status_code, resp.json())
        glossaries.append(resp.json())
    return glossaries


def get_glossary_ids(auth_key):
    url = "https://api.deepl.com/v3/glossaries"
    headers = {
        "Authorization": f"DeepL-Auth-Key {auth_key}",
        "User-Agent": "YourApp/1.2.3",
    }
    resp = requests.get(url, headers=headers, timeout=30)
    print(resp.status_code)
    return [x.get("glossary_id") for x in resp.json()["glossaries"]]


def transalate_to(text_l, s_lang, t_lang, glossary_id):
    res = []
    for text in text_l:
        if glossary_id:
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang, glossary=glossary_id).text)
        else: 
            res.append(deepl_client.translate_text(text, target_lang=t_lang, source_lang=s_lang).text)
    return res


In [12]:
generate_glossaries(["config_files/ger_to_eng.csv", "config_files/eng_to_ger.csv"], ["DE","EN"], ["EN", "DE"], deepl_key)
# get all current glossaries
glossary_ids = get_glossary_ids(deepl_key)

201 {'glossary_id': 'd66b46af-d866-4ed3-b208-3742e7508280', 'name': 'glossary0', 'dictionaries': [{'source_lang': 'de', 'target_lang': 'en', 'entry_count': 5}], 'creation_time': '2026-01-05T15:51:58.411147Z'}
400 {'message': 'Bad request. Reason: Invalid entries'}
200


In [13]:
# translate ki terms as test
'''
test_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
print(test_terms)
translations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])
print(translations)
'''

'\ntest_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())\nprint(test_terms)\ntranslations = transalate_to(test_terms, "DE", "EN-GB", glossary_ids[0])\nprint(translations)\n'

In [14]:
# write text (from file) and its transaltion to df and save to csv and json file
def tranlate_to_df(txts_json, file_to):
    # load json as dict
    with open(txts_json, "r") as f:
        txts = json.load(f)

    res_df = pd.DataFrame([(key, var) for (key, L) in txts.items() for var in L], columns=['id', 'text']).drop_duplicates(subset=['text']).reset_index(drop=True)
    #fill df with traslations
    res_df['translation'] = transalate_to(res_df['text'].tolist(), "DE", "EN-GB", glossary_ids[0]) # TODO: changing source and target langs dependeing on input
    #save to file 
    res_df.to_csv(f"{file_to}.csv", index=False)
    res_df.to_json(f"{file_to}.json", orient="records")
    return res_df


### Translate KI_sentences to English

In [15]:
#tranlate_to_df("generated_data/KI_sentences.json", "generated_data/KI_sentences_translated")

In [16]:
tranlate_to_df("generated_data/KI_sentences_full.json", "generated_data/KI_sentences_full_translated")

,id,text,translation
0,1739352_full.txt,Mit einer Mahnwache gegen die Todesstrafe und ...,With a vigil against the death penalty and a s...
1,1364384_full.txt,Berlin taz – Ein edleres Spendensiegel ist kau...,Berlin taz - A nobler seal of approval for don...
2,483913_full.txt,930 Friedrichstadtpalast Eu cand vreau sa flui...,930 Friedrichstadtpalast Eu cand vreau sa flui...
3,1305430_full.txt,München tazdpa – Die Mahnwache mahnte die Spru...,Munich tazdpa - The vigil warned the banners o...
4,1573221_full.txt,Berlin taz – Schon am kommenden Donnerstag wer...,"Berlin taz - The first refugees from Serbia, i..."
...,...,...,...
145,1308758_full.txt,Markiert die Verabschiedung der „Allgemeinen E...,"Does the adoption of the ""Universal Declaratio..."
146,5319720_full.txt,von Alke Wierth Doch es gibt sie die Mo­men­te...,by Alke Wierth But there are moments when the ...
147,791729_full.txt,von HEIKE KLEFFNER „Ich müsste lügen wenn ich ...,"by HEIKE KLEFFNER ""I would have to lie if I sa..."
148,5321401_full.txt,Für die spanische Originalversion bitte herunt...,"For the original Spanish version, please scrol..."


In [17]:
# input to anthroscore
# get AI-related words, sorted according to 'Häufigkeitsklassen' and from AnthroScore
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
ki_terms += ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung']
ai_terms = transalate_to(ki_terms, "DE", "EN-GB", glossary_ids[0])

In [18]:
with open(f"config_files/ai_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ai_terms])
        f.close()

with open(f"config_files/ki_terms.txt", "w") as f:
        f.writelines([" "+f"'{x}'" for x in ki_terms])
        f.close()